# PROG8431 – Solutions to Highway Congestion

**Team:**
- Jasper Lim (917038)

- John Buni (9115726)

- Eche Oji (9078881)

## Problem & Research Question
We are exploring traffic congestion on provincial highways in Southern Ontario.

**Research question:** What factors contribute to highway congestion, and can traffic data help identify high-congestion periods or alternative routes?

### Scope
This notebook focuses on **2024 AADT data exploration only**. It does not perform prediction. Since the dataset contains one year of AADT, historical trends require additional data.

## Questions Considered for Choosing Data

- **One-time or ongoing?** One-time analysis of the 2024 data; the same approach can be reused with future data.
- **Automated?** Not currently, but the Python workflow can be reused or automated with new data.
- **How can we track it?** Compare **AADT** across highway segments to track traffic volume.

## Data Source

We use the **Provincial Highways Traffic Volumes 2024 AADT** dataset provided to the team.

**Key fields:** Highway, segment start/end locations, Distance (KM), and 2024 AADT.

Source area: Ontario Ministry of Transportation traffic-volume portal:  
https://www.library.mto.gov.on.ca/SydneyPLUS/TechPubs/Portal/tp/tvSplash.aspx

*Future work can combine this with Ontario 511 incident/real-time data.*

## Setup

Required packages: `pandas`, `numpy`, `matplotlib`, and `matplotlib-venn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load the Dataset

The CSV is stored in the project's `data` folder.

In [ ]:
DATA_PATH = Path("data/Provincial Highways Traffic Volumes 2024 AADT Only (1).csv")

data = pd.read_csv(DATA_PATH)

print("Original dataset shape:", data.shape)
display(data.head())

## Understand the Data

Each row represents a highway segment. The main variables are **Distance (KM)** and **2024 AADT**. Highway and location fields allow segment comparisons.

**AADT measures traffic volume, not congestion directly.**


The dataset describes Ontario provincial highway segments, their locations, distance, and 2024 AADT. It contains **2,712 rows before cleansing** and about **1,800 usable rows** after removing blank/incomplete records.

It supports comparison of traffic volumes and segment distances, but one year of AADT cannot show historical trends or predict future congestion.

In [ ]:
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)

print("\nMissing values:")
display(data.isna().sum())

print("\nNumber of duplicate rows:", data.duplicated().sum())

## Data Cleansing

We check for blank rows, duplicates, inconsistent text, incorrect numeric types, missing values, and invalid numerical values.

For this dataset, we remove blank separator rows and duplicates, trim text, convert Distance and AADT to numeric values, remove records missing either value, and exclude invalid distances or negative AADT.

In [ ]:
# Clean the data
cleaned_data = data.copy()

cleaned_data = cleaned_data.dropna(how="all").copy()

text_columns = [
    "Highway",
    "Location Description From",
    "Location Description To"
]

for column in text_columns:
    cleaned_data[column] = cleaned_data[column].astype("string").str.strip()

numeric_columns = ["Distance (KM)", "2024 AADT"]

for column in numeric_columns:
    cleaned_data[column] = pd.to_numeric(
        cleaned_data[column],
        errors="coerce"
    )

cleaned_data = cleaned_data.drop_duplicates()

cleaned_data = cleaned_data.dropna(
    subset=["Distance (KM)", "2024 AADT"]
)

cleaned_data = cleaned_data[
    (cleaned_data["Distance (KM)"] > 0) &
    (cleaned_data["2024 AADT"] >= 0)
].copy()

cleaned_data = cleaned_data.reset_index(drop=True)

print("Original rows:", len(data))
print("Rows after cleansing:", len(cleaned_data))
print("Rows removed:", len(data) - len(cleaned_data))

display(cleaned_data.head())

In [ ]:
print("Cleaned dataset shape:", cleaned_data.shape)

print("\nMissing values after cleansing:")
display(cleaned_data.isna().sum())

print("\nDuplicate rows after cleansing:", cleaned_data.duplicated().sum())

print("\nNumber of unique highways represented:")
print(cleaned_data["Highway"].nunique())

print("\nNumerical summary after cleansing:")
display(cleaned_data[["Distance (KM)", "2024 AADT"]].describe())

## Challenge 4 – Classes and Methods

`HighwayTrafficAnalyzer` groups the main calculations so the analysis can be reused with the cleaned dataset.

In [ ]:
class HighwayTrafficAnalyzer:
    """Analyze Ontario provincial highway traffic-volume data."""

    def __init__(self, data):
        """Store a copy of the analysis dataset."""
        self.data = data.copy()

    def calculate_basic_statistics(self, column):
        """Calculate mean, median, and mode for a numeric column."""
        series = self.data[column]

        return {
            "mean": series.mean(),
            "median": series.median(),
            "mode": series.mode().tolist()
        }

    def calculate_variability(self, column):
        """Calculate variance, standard deviation, and four percentile values."""
        series = self.data[column]

        return {
            "variance": series.var(),
            "standard_deviation": series.std(),
            "quartiles": series.quantile([0.25, 0.50, 0.75, 1.00])
        }

    def calculate_highway_averages(self):
        """Calculate average distance and AADT for each highway."""
        return self.data.groupby("Highway").agg(
            average_distance=("Distance (KM)", "mean"),
            average_aadt=("2024 AADT", "mean")
        )

    def calculate_highway_variability(self):
        """Calculate variance and standard deviation by highway."""
        return self.data.groupby("Highway").agg(
            distance_variance=("Distance (KM)", "var"),
            distance_std=("Distance (KM)", "std"),
            aadt_variance=("2024 AADT", "var"),
            aadt_std=("2024 AADT", "std")
        )

    def get_median_groups(self):
        """Return segment index sets above the median distance and AADT."""
        distance_median = self.data["Distance (KM)"].median()
        aadt_median = self.data["2024 AADT"].median()

        long_sections = set(
            self.data.index[self.data["Distance (KM)"] > distance_median]
        )

        high_traffic_sections = set(
            self.data.index[self.data["2024 AADT"] > aadt_median]
        )

        return long_sections, high_traffic_sections

In [ ]:
analyzer = HighwayTrafficAnalyzer(cleaned_data)

highway_averages = analyzer.calculate_highway_averages()
highway_variability = analyzer.calculate_highway_variability()

print("Average distance and AADT by highway:")
display(highway_averages.head())

print("Variability by highway:")
display(highway_variability.head())

## Challenge 5 – Mean, Median, and Mode

We calculate mean, median, and mode for **Distance (KM)** and **2024 AADT**. Comparing mean and median helps show whether unusually high values affect the average.

In [ ]:
distance_stats = analyzer.calculate_basic_statistics("Distance (KM)")
aadt_stats = analyzer.calculate_basic_statistics("2024 AADT")

print("Distance (KM)")
print(f"  Mean:   {distance_stats['mean']:,.2f}")
print(f"  Median: {distance_stats['median']:,.2f}")
print(f"  Mode:   {distance_stats['mode']}")

print("\n2024 AADT")
print(f"  Mean:   {aadt_stats['mean']:,.2f}")
print(f"  Median: {aadt_stats['median']:,.2f}")
print(f"  Mode:   {aadt_stats['mode']}")

## Challenge 6 – Variability and Quartiles

We calculate variance, standard deviation, and the four requested percentile points: **Q1 (25%), Q2/median (50%), Q3 (75%), and Q4/maximum (100%)**.

We also compare variability across highways.

In [ ]:
distance_variability = analyzer.calculate_variability("Distance (KM)")
aadt_variability = analyzer.calculate_variability("2024 AADT")

quartile_labels = {
    0.25: "Q1 (25%)",
    0.50: "Q2 / Median (50%)",
    0.75: "Q3 (75%)",
    1.00: "Q4 / Maximum (100%)"
}

print("Distance (KM)")
print(f"  Variance:           {distance_variability['variance']:,.2f}")
print(f"  Standard deviation: {distance_variability['standard_deviation']:,.2f}")
print("  Quartiles:")
print(distance_variability["quartiles"].rename(quartile_labels).to_string())

print("\n2024 AADT")
print(f"  Variance:           {aadt_variability['variance']:,.2f}")
print(f"  Standard deviation: {aadt_variability['standard_deviation']:,.2f}")
print("  Quartiles:")
print(aadt_variability["quartiles"].rename(quartile_labels).to_string())

In [ ]:
highway_variability = analyzer.calculate_highway_variability()

display(highway_variability)

## Challenge 7 – Scatter Plot

The scatter plot compares **Distance (KM)** with **2024 AADT** to explore whether segment length and traffic volume appear related.

In [ ]:
plt.figure(figsize=(9, 6))

for group, group_data in cleaned_data.groupby("Highway"):
    plt.scatter(
        group_data["Distance (KM)"],
        group_data["2024 AADT"],
        alpha=0.5
    )

plt.xlabel("Distance (KM)")
plt.ylabel("2024 AADT")
plt.title("Highway Segment Distance vs 2024 Traffic Volume")
plt.grid(True, alpha=0.2)
plt.show()

## Challenge 7 – Histograms

Histograms show the distributions of **Distance (KM)** and **2024 AADT**.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(cleaned_data["Distance (KM)"], bins=15, edgecolor="black")
plt.xlabel("Distance (KM)")
plt.ylabel("Frequency")
plt.title("Distribution of Highway Segment Distance")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(cleaned_data["2024 AADT"], bins=15, edgecolor="black")
plt.xlabel("2024 AADT")
plt.ylabel("Frequency")
plt.title("Distribution of 2024 AADT")
plt.show()

## Challenge 7 – Box-and-Whisker Plots

Boxplots compare the spread and potential outliers in distance and AADT across highways.

In [ ]:
plt.figure(figsize=(12, 6))
cleaned_data.boxplot(column="Distance (KM)", by="Highway")
plt.title("Highway Segment Distance by Highway")
plt.suptitle("")
plt.xlabel("Highway")
plt.ylabel("Distance (KM)")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 6))
cleaned_data.boxplot(column="2024 AADT", by="Highway")
plt.title("2024 AADT by Highway")
plt.suptitle("")
plt.xlabel("Highway")
plt.ylabel("2024 AADT")
plt.xticks(rotation=45)
plt.show()

## Challenge 7 – Venn Diagram

The Venn diagram compares segments above the median for **Distance** and **AADT**. The overlap shows segments above both medians.

This is descriptive only; **above-median AADT does not mean a segment is congested**.

In [ ]:
long_sections, high_traffic_sections = analyzer.get_median_groups()

plt.figure(figsize=(8, 6))

venn2(
    [long_sections, high_traffic_sections],
    set_labels=("Above-Median Distance", "Above-Median AADT")
)

plt.title("Highway Segments Above the Median for Distance and AADT")
plt.show()

print("Segments above median distance:", len(long_sections))
print("Segments above median AADT:", len(high_traffic_sections))
print("Segments above both medians:", len(
    long_sections.intersection(high_traffic_sections)
))

## Challenge 7 – Numerical Summary

`describe()` summarizes the count, mean, standard deviation, minimum, quartiles, and maximum for Distance and AADT.

In [ ]:
numerical_summary = cleaned_data[
    ["Distance (KM)", "2024 AADT"]
].describe()

display(numerical_summary)

## Findings & Conclusion

The analysis describes how highway-segment distance and 2024 traffic volume vary across Ontario highways. The statistics and visualizations help compare segments and identify higher-volume observations.

However, AADT alone cannot determine congestion or its causes. Future work should add speed, travel time, capacity, incidents, weather, time-of-day, and historical traffic data.

## Use-Case Summary

This dataset contains 2024 annual average daily traffic volumes for segments of provincial highways. It includes highway identifiers, segment locations, segment distance, and traffic volume. Exploratory analysis can identify high-volume segments, describe traffic-volume distributions, compare highway segments, and examine whether segment distance is associated with traffic volume across Ontario highways.